# Deep SRQ Solver Ablation

Runs DeepSRQ-vs-DeepSRQ configurations per scenario using serial rollouts, a pooled PATH solver backend, and pure starts plus five random restarts.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "bimatrix_game":
    BIMATRIX_DIR = ROOT
else:
    BIMATRIX_DIR = ROOT / "discrete_action_space" / "bimatrix_game"
DISCRETE_DIR = BIMATRIX_DIR.parent
for path in (BIMATRIX_DIR, DISCRETE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from experiment_harness import (
    BASE_SEED,
    configure_path_runtime,
    run_deep_srq_ablation_variants,
    summarize_ablation_timing_rows,
)
from stats_utils import save_training_stats


In [2]:
SCENARIOS = ("scenario1", "scenario3")
N_EPISODES = 3000
PATHWRAP = str(configure_path_runtime(DISCRETE_DIR))
OUTPUT_ROOT = BIMATRIX_DIR / "ablation_runs" / "solver_ablation"
USE_GPU = True
SOLVER_NAME = "path_c_pool"
BASE_HP = {
    "sre_solver_workers": 8,
    "sre_num_repeats": 5,
    "sre_include_pure_starts": True,
}

RUN_VARIANTS = (
    {"label": "path_pool8_pure_plus_random5_eps0.25_linear", "epsilon_robust_initial": 0.25, "epsilon_schedule": "linear"},
    {"label": "path_pool8_pure_plus_random5_eps0.5_linear", "epsilon_robust_initial": 0.5, "epsilon_schedule": "linear"},
)

In [3]:
results = run_deep_srq_ablation_variants(
    variants=RUN_VARIANTS,
    base_seed=BASE_SEED,
    pathwrap_path=PATHWRAP,
    scenarios=SCENARIOS,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    write_plots=False,
    default_n_episodes=N_EPISODES,
    default_solver_name=SOLVER_NAME,
    default_hyperparameter_overrides=BASE_HP,
)

save_training_stats(OUTPUT_ROOT / "deep_srq_solver_ablation_manifest.txt", results)

for row in summarize_ablation_timing_rows(results):
    print(row)

Deep SRQ Scenario 1 | DeepSRQ vs DeepSRQ | eps0=0.25 | schedule=linear | solver=path_c_pool | seed=2025


scenario1:eps0.25_linear__path_pool8_pure_plus_random5_eps0.25_linear:path_c_pool:   3%|▎         | 88/3000 [00:39<21:50,  2.22it/s]


KeyboardInterrupt: 

In [ ]:
def _fmt(value, kind):
    if value is None:
        return "-"
    if kind == "int":
        return f"{int(value):,}"
    if kind == "float1":
        return f"{float(value):,.1f}"
    if kind == "float2":
        return f"{float(value):,.2f}"
    return str(value)


def print_pretty_table(rows, columns):
    rendered = []
    headers = [title for _, title, _ in columns]
    for row in rows:
        rendered.append([_fmt(row.get(key), kind) for key, _, kind in columns])
    widths = [len(header) for header in headers]
    for row in rendered:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    header_line = " | ".join(header.ljust(width) for header, width in zip(headers, widths))
    separator = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(separator)
    for row in rendered:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))


CASE_LABELS = {
    "path_pool8_pure_plus_random5_eps0.25_linear": "PATH pool(8) | pure + 5 random | eps0=0.25",
    "path_pool8_pure_plus_random5_eps0.5_linear": "PATH pool(8) | pure + 5 random | eps0=0.5",
}


summary_rows = []
for row in summarize_ablation_timing_rows(results):
    stats = results[row["scenario"]][row["variant"]]
    summary_rows.append({
        **row,
        "case": CASE_LABELS.get(row["variant"], row["variant"]),
        "epsilon": stats["epsilon_robust_initial"],
        "schedule": stats["epsilon_schedule"],
        "repeats": stats["hyperparameters"].get("sre_num_repeats"),
        "pure_starts": stats["hyperparameters"].get("sre_include_pure_starts"),
        "solver": stats.get("solver_name"),
    })

summary_rows.sort(key=lambda row: (row["scenario"], row["epsilon"], row["case"]))

print_pretty_table(
    summary_rows,
    columns=(
        ("scenario", "Scenario", "str"),
        ("epsilon", "Eps0", "float2"),
        ("schedule", "Schedule", "str"),
        ("solver", "Solver", "str"),
        ("pure_starts", "Pure", "str"),
        ("repeats", "Repeats", "int"),
        ("env_steps", "Env Steps", "int"),
        ("wall_seconds", "Wall s", "float1"),
        ("steps_per_second", "Steps/s", "float2"),
        ("mean_sre_ms", "SRE ms", "float2"),
        ("mean_backend_ms", "Backend ms", "float2"),
        ("agent1_mean_last", "A1 Last", "float2"),
        ("agent2_mean_last", "A2 Last", "float2"),
    ),
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def rolling_mean(values, window=100):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(values, kernel, mode="valid")


ROLLING_WINDOW = 100

for scenario_key in SCENARIOS:
    fig, ax = plt.subplots(figsize=(9, 5))
    for variant in RUN_VARIANTS:
        stats = results[scenario_key][variant["label"]]
        rewards = stats["rewards"]
        min_len = min(len(rewards[0]), len(rewards[1]))
        if min_len == 0:
            continue
        joint_reward = np.asarray(rewards[0][:min_len], dtype=float) + np.asarray(rewards[1][:min_len], dtype=float)
        smoothed = rolling_mean(joint_reward, window=ROLLING_WINDOW)
        x = np.arange(smoothed.size) + min(ROLLING_WINDOW, joint_reward.size)
        ax.plot(x, smoothed, label=CASE_LABELS.get(variant["label"], variant["label"]))
    ax.set_title(f"{scenario_key} | serial | path_c_pool | pure + 5 random")
    ax.set_xlabel("Completed episodes")
    ax.set_ylabel(f"Joint reward, rolling mean ({ROLLING_WINDOW})")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.savefig(OUTPUT_ROOT / f"reward_curves_{scenario_key}.png")
    plt.show()
